# Exploring Subjecthood in Multilingual Models

Code for explorative analysis of the UD data from the graduation thesis 'Exploring Subjecthood in Multilingual Language Models'.

## Installations & Imports

In this Section, all needed modules and libraries are downloaded and imported.

In order to run code and save fine-tuned models, the token from [Hugging Face account](https://huggingface.co/) is needed.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
%pip install -U kaleido

In [ ]:
%pip install plotly==6.1.1 pyconll transformers datasets evaluate seqeval huggingface_hub scikit-learn scipy

In [ ]:
!apt update
!apt install -y chromium-browser chromium-chromedriver \
libnss3 libatk-bridge2.0-0 libcups2 libxcomposite1 libxdamage1 \
libxfixes3 libxrandr2 libgbm1 libxkbcommon0 libpango-1.0-0 libcairo2 libasound2

In [ ]:
import kaleido
import os
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

In [ ]:
import pyconll
import statistics
from pprint import pprint
import numpy as np
from collections import Counter
import tqdm
import json
import random

In [ ]:
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from torch.utils.data import DataLoader

import evaluate
from sklearn.metrics import accuracy_score, classification_report, hamming_loss
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from huggingface_hub import notebook_login
from scipy.stats import wilcoxon

In [ ]:
kaleido.get_chrome_sync()

In [ ]:
os.environ["KALIEDO_PATH"] = "/usr/bin/chromium-browser"
os.environ["CHROMIUM_PATH"] = "/usr/bin/chromium-browser"

In [ ]:
notebook_login()

In [ ]:
!pip freeze > requirements.txt

## Data Download

In this Section, the data from [Universal Dependencies](https://universaldependencies.org/) is downloaded.

In [ ]:
%mkdir data
%cd data

In [ ]:
data = {
    "Arabic": {
        "treebank": "UD_Arabic-PADT",
        "path": "/content/data/Arabic/ar_padt-ud",
        "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_Arabic-PADT/refs/heads/master/ar_padt-ud-train.conllu",
        "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_Arabic-PADT/refs/heads/master/ar_padt-ud-dev.conllu",
        "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_Arabic-PADT/refs/heads/master/ar_padt-ud-test.conllu",
        "model": "fromdeath2morning/mbert-argumentClassification-arabic",
    },
    "Basque": {
        "treebank": "UD_Basque-BDT",
        "path": "/content/data/Basque/eu_bdt-ud",
        "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_Basque-BDT/refs/heads/master/eu_bdt-ud-train.conllu",
        "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_Basque-BDT/refs/heads/master/eu_bdt-ud-dev.conllu",
        "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_Basque-BDT/refs/heads/master/eu_bdt-ud-test.conllu",
        "model": "fromdeath2morning/mbert-argumentClassification-basque",
    },
    "English": {
        "treebank": "UD_English-EWT",
        "path": "/content/data/English/en_ewt-ud",
        "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/refs/heads/master/en_ewt-ud-train.conllu",
        "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/refs/heads/master/en_ewt-ud-dev.conllu",
        "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/refs/heads/master/en_ewt-ud-test.conllu",
        "model": "fromdeath2morning/mbert-argumentClassification-english",
    },
    "German": {
        "treebank": "UD_German-GSD",
        "path": "/content/data/German/de_gsd-ud",
        "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_German-GSD/refs/heads/master/de_gsd-ud-train.conllu",
        "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_German-GSD/refs/heads/master/de_gsd-ud-dev.conllu",
        "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_German-GSD/refs/heads/master/de_gsd-ud-test.conllu",
        "model": "fromdeath2morning/mbert-argumentClassification-german",
    },
    "Hindi": {
        "treebank": "UD_Hindi-HDTB",
        "path": "/content/data/Hindi/hi_hdtb-ud",
        "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_Hindi-HDTB/refs/heads/master/hi_hdtb-ud-train.conllu",
        "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_Hindi-HDTB/refs/heads/master/hi_hdtb-ud-dev.conllu",
        "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_Hindi-HDTB/refs/heads/master/hi_hdtb-ud-test.conllu",
        "model": "fromdeath2morning/mbert-argumentClassification-hindi",
    },
    "Russian": {
        "treebank": "UD_Russian-GSD",
        "path": "/content/data/Russian/ru_gsd-ud",
        "train": "https://github.com/UniversalDependencies/UD_Russian-GSD/raw/refs/heads/master/ru_gsd-ud-train.conllu",
        "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-GSD/refs/heads/master/ru_gsd-ud-dev.conllu",
        "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-GSD/refs/heads/master/ru_gsd-ud-test.conllu",
        "model": "fromdeath2morning/mbert-argumentClassification-russian",
    },
    "Turkish": {
        "treebank": "UD_Turkish-Penn",
        "path": "/content/data/Turkish/tr_penn-ud",
        "train": "https://github.com/UniversalDependencies/UD_Turkish-Penn/raw/refs/heads/master/tr_penn-ud-train.conllu",
        "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_Turkish-Penn/refs/heads/master/tr_penn-ud-dev.conllu",
        "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_Turkish-Penn/refs/heads/master/tr_penn-ud-test.conllu",
        "model": "fromdeath2morning/mbert-argumentClassification-turkish",
    },
    "Welsh": {
        "treebank": "UD_Welsh-CCG",
        "path": "/content/data/Welsh/cy_ccg-ud",
        "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_Welsh-CCG/refs/heads/master/cy_ccg-ud-train.conllu",
        "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_Welsh-CCG/refs/heads/master/cy_ccg-ud-dev.conllu",
        "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_Welsh-CCG/refs/heads/master/cy_ccg-ud-test.conllu",
        "model": "fromdeath2morning/mbert-argumentClassification-welsh",
    },
}

In [ ]:
for language in data:
    %mkdir "{language}"
    %cd "{language}"
    for mode in ["train", "dev", "test"]:
        !wget "{data[language][mode]}"
    %cd ..

## Argument's Distribution

In this Section, I analyze the distribution of `S`, `A` and `P` arguments per pos in UD data for $8$ chosen languages.

In [ ]:
def rolesCount(path, targetPos):
    """
    This function calculates the amount of S-, A-, P-arguments of the target part-of-speech in data.

    :param path: the path to the conllu-file
    :param targetPos: list with target parts of speech
    :returns: the dict with calculated statistics of the amount of the arguments in data
    """
    stats = {"S": 0, "A": 0, "P": 0}

    text = pyconll.load_from_file(path)

    for sentence in text:
        for token in sentence:
            if token.upos == "VERB":
                head = token
                childrenS = [
                    int(token.id)
                    for token in sentence
                    if token.head == head.id
                    and token.upos in targetPos
                    and "nsubj" in token.deprel
                ]
                childrenP = [
                    int(token.id)
                    for token in sentence
                    if token.head == head.id
                    and token.upos in targetPos
                    and "obj" in token.deprel
                ]

                if len(childrenP) > 0:
                    stats["A"] += len(childrenS)
                    stats["P"] += len(childrenP)
                else:
                    stats["S"] += len(childrenS)

    return stats

In [ ]:
for language in data:
    print(f"---- {language.upper()} ----")
    path = data[language]["path"]
    for mode in ["train", "test"]:
        print(mode, "nouns")
        pprint(rolesCount(f"{path}-{mode}.conllu", ["NOUN", "PROPN"]))
        print(mode, "pronouns")
        pprint(rolesCount(f"{path}-{mode}.conllu", ["PRON"]))
    print()

## Word Order Distribution

In this Section, I analyze the word order distribution in UD data for $8$ chosen languages, visualize it in bar charts and calculate the median and mean sentence lengths.

In [ ]:
def distanceCount(path, targetPos):
    """
    This function calculates the distance between S-, A-, P-arguments and their heads.

    :param path: the path to the conllu-file
    :param targetPos: list with target parts of speech
    :returns: the amount of sentences in corpus, the list of sentences' lengths and three lists (for S-, A- and P- arguments) with calculated distanced for each noun that was found in conllu
    """
    sentenceCount = 0
    sentenceLengths = []

    argumentA = []
    argumentS = []
    argumentP = []

    text = pyconll.load_from_file(path)

    for sentence in text:
        sentenceCount += 1
        tokenCount = 0
        for token in sentence:
            tokenCount += 1
            if token.upos == "VERB":
                head = token
                childrenS = [
                    int(token.id)
                    for token in sentence
                    if token.head == head.id
                    and token.upos in targetPos
                    and "nsubj" in token.deprel
                ]
                childrenP = [
                    int(token.id)
                    for token in sentence
                    if token.head == head.id
                    and token.upos in targetPos
                    and "obj" in token.deprel
                ]

                if len(childrenP) > 0:
                    for id_ in childrenS:
                        argumentA.append(id_ - int(head.id))

                    for id_ in childrenP:
                        argumentP.append(id_ - int(head.id))
                else:
                    for id_ in childrenS:
                        argumentS.append(id_ - int(head.id))

        sentenceLengths.append(tokenCount)

    return sentenceCount, sentenceLengths, argumentA, argumentS, argumentP

In [ ]:
%cd ..

In [ ]:
stats = {}
for language in data:
    path = data[language]["path"]
    stats[language] = {}

    sentenceCount, sentenceLengths, argumentA, argumentS, argumentP = 0, [], [], [], []

    for mode in ["train", "dev", "test"]:
        cnt, lens, cntA, cntS, cntP = distanceCount(f"{path}-{mode}.conllu", ["NOUN", "PROPN", "PRON"])

        sentenceCount += cnt
        sentenceLengths += lens
        argumentA += cntA
        argumentS += cntS
        argumentP += cntP

    stats[language] = {
        "treebank size": sentenceCount,
        "avg sentence length": statistics.mean(sentenceLengths),
        "median sentence length": statistics.median(sentenceLengths),
    }

    fig = make_subplots(
        rows=1, cols=3, subplot_titles=["S-argument", "A-argument", "P-argument"]
    )

    fig.add_trace(
        go.Bar(
            x=sorted(set(argumentS)),
            y=[argumentS.count(value) for value in sorted(set(argumentS))],
            marker_color="#7201a8",
            showlegend=False,
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Bar(
            x=sorted(set(argumentA)),
            y=[argumentA.count(value) for value in sorted(set(argumentA))],
            marker_color="#9c179e",
            showlegend=False,
        ),
        row=1,
        col=2,
    )

    fig.add_trace(
        go.Bar(
            x=sorted(set(argumentP)),
            y=[argumentP.count(value) for value in sorted(set(argumentP))],
            marker_color="#bd3786",
            showlegend=False,
        ),
        row=1,
        col=3,
    )

    for i in range(1, 4):
        fig.add_vline(
            x=0, line_width=1, line_color="#0d0887", line_dash="dash", row=1, col=i
        )

    fig.update_layout(
        font_family="Brill",
        font_color="black",
        title_font_family="Brill",
        template="plotly_white",
        width=1200,
        height=400,
        margin=dict(l=10, r=10, t=10, b=40),
        # title_text=f"The distances between S-, A- and P-arguments and their heads in {language}"
    )

    for i, annotation in enumerate(fig.layout.annotations):
        annotation.update(
            y=-0.05,
            yanchor='top',
            font=dict(size=16)
        )

    fig.show()
    pio.write_image(fig, f"The distances between S-, A- and P-arguments and their heads in {language}.png", width=1200, height=400, scale=2)

In [ ]:
x = [stats[language]["avg sentence length"] for language in stats]
y = [stats[language]["median sentence length"] for language in stats]
languages = list(stats.keys())

fig = px.scatter(
    x=x,
    y=y,
    color=languages,
    color_discrete_sequence=[
        "#9c179e",
        "#0d0887",
        "#fca636",
        "#bd3786",
        "#46039f",
        "#ed7953",
        "#7201a8",
        "#d8576b",
    ],
)

colors = {lang: fig.data[i].marker.color for i, lang in enumerate(languages)}

cnt = 0
for x, y in zip(x, y):
    fig.add_shape(
        type="line",
        x0=x,
        y0=0,
        x1=x,
        y1=y,
        line=dict(color=colors[languages[cnt]], width=1, dash="dot"),
    )

    fig.add_shape(
        type="line",
        x0=0,
        y0=y,
        x1=x,
        y1=y,
        line=dict(color=colors[languages[cnt]], width=1, dash="dot"),
    )

    fig.add_annotation(
        x=x,
        y=0,
        text=languages[cnt],
        showarrow=False,
        yref="paper",
        xanchor="right",
        yanchor="top",
        textangle=-45,
        font=dict(color=colors[languages[cnt]], size=12),
        xshift=10,
        yshift=-5,
    )

    cnt += 1

fig.update_layout(
    font_family="Brill",
    font_color="black",
    title_font_family="Brill",
    template="plotly_white",
    margin=dict(l=10, t=10, b=20, r=10),
    xaxis_title=dict(
        text="Average Sentence Length",
        standoff=40,
    ),
    yaxis_title="Median Sentence Length",
    showlegend=False,
)

fig.show()
pio.write_image(
    fig, "Average vs Median Sentence Length.png", width=1000, height=600, scale=2
)

## Morphology Analysis

In this Section, I analyze which morphological features are most common for the core arguments (`S`, `A` and `P`) in UD data.


In [ ]:
def roleAnalysis(path, targetPos, argument):
    """
    This function extracts the features that a clause argument (S, A or P) has in conllu texts.
    :param path: the path to the conllu file
    :param targetPos: list with the target parts of speech
    :param argument: the target role
    :returns: the list of tuples, where each one is a tuple of features that a target argument can have in conllu texts
    """

    result = []

    text = pyconll.load_from_file(path)

    for sentence in text:
        for token in sentence:
            if token.upos == "VERB":
                head = token

                if argument == "P":
                    feature = []
                    for token in sentence:
                        if (
                            token.head == head.id
                            and token.upos in targetPos
                            and "obj" in token.deprel
                        ):
                            for value in token.feats.values():
                                feature += list(value)
                            result.append(tuple(feature))

                hasObj = (
                    len(
                        [
                            int(token.id)
                            for token in sentence
                            if token.head == head.id
                            and token.upos in targetPos
                            and "obj" in token.deprel
                        ]
                    )
                    > 0
                )

                if argument == "S":
                    feature = []
                    for token in sentence:
                        if (
                            token.head == head.id
                            and token.upos in targetPos
                            and "nsubj" in token.deprel
                            and not hasObj
                        ):
                            for value in token.feats.values():
                                feature += list(value)
                            result.append(tuple(feature))

                if argument == "A":
                    feature = []
                    for token in sentence:
                        if (
                            token.head == head.id
                            and token.upos in targetPos
                            and "nsubj" in token.deprel
                            and hasObj
                        ):
                            for value in token.feats.values():
                                feature += list(value)
                            result.append(tuple(feature))
    return result

In [ ]:
for language in data:
    print(f"---- {language.upper()} ----")
    path = data[language]["path"]
    for mode in ["train", "test"]:
        for role in ["S", "A", "P"]:
            targetRole = Counter(roleAnalysis(f"{path}-{mode}.conllu", ["NOUN", "PROPN"], role))
            print(f"mode: {mode}\nrole: {role} argument\npos: NOUN, PROPN")
            pprint(targetRole.most_common(3))
            print()

            targetRole = Counter(roleAnalysis(f"{path}-{mode}.conllu", ["PRON"], role))
            print(f"pos: PRON")
            pprint(targetRole.most_common(3))
            print()
            print()

## Mistakes Analysis

In this Section, I analyze what mistakes the models have done on the sample with non dominant word order. In particular, I visualize the length of the sentences where the mistakes were made and compare the models' results.

In [ ]:
with open("analyzing non-dominant word order - mBERT.tsv", "r") as file:
    file = file.readlines()
    data1 = [int(line.split("\t")[3]) for line in file[1:]]
    mistakes1 = set([(line.split("\t")[2], line.split("\t")[5]) for line in file[1:]])

with open("analyzing non-dominant word order - XLM-R.tsv", "r") as file:
    file = file.readlines()
    data2 = [int(line.split("\t")[3]) for line in file[1:]]
    mistakes2 = set([(line.split("\t")[2], line.split("\t")[5]) for line in file[1:]])

fig = make_subplots(
    rows=1,
    cols=2,
)

fig.add_trace(
    go.Violin(
        y=data1,
        name="multilingual BERT",
        line_color="#9c179e",
        fillcolor="#9c179e",
        opacity=0.6,
        line_width=2,
        box_visible=True,
        meanline_visible=True,
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Violin(
        y=data2,
        name="XLM-RoBERTa",
        line_color="#bd3786",
        fillcolor="#bd3786",
        line_width=2,
        opacity=0.6,
        box_visible=True,
        meanline_visible=True,
    ),
    row=1,
    col=2
)

fig.update_layout(
    font_family="Brill",
    font_color="black",
    font_size=14,
    title_font_family="Brill",
    template="plotly_white",
    width=700,
    height=700,
    margin=dict(l=10, r=10, t=20, b=10),
    showlegend=False,
)

fig.update_yaxes(title_text="Sentence length", row=1, col=1)

fig.show()

pio.write_image(fig, f"violins_sentence_length.png", width=700, height=700, scale=2)

## Statistical Analysis

In this Section, I aim to compare the layerwise distributions of f1-scores per each class (`S`, `A` and `P`) computed during the probing of the `mBERT` and the `XLM-RoBERTa`. In this way, I want to understand whether the distribution of values differs significantly or, in other words, does one model «knows» more than another one.

In [ ]:
for language in data:
    with open(f"/content/mBERT/metrics_{language}_per_layer.json", "r") as file:
        mBERTdata = json.load(file)

    with open(f"/content/XLM-RoBERTa/metrics_{language}_per_layer.json", "r") as file:
        XLMdata = json.load(file)

    print(f"----{language}----")
    for arg in ["S", "A", "P"]:
        print(f"{arg}. ", end="")
        x = [mBERTdata[f"layer {layer}"][arg]["f1-score"] for layer in range(13)]
        y = [XLMdata[f"layer {layer}"][arg]["f1-score"] for layer in range(13)]
        test = wilcoxon(x, y, alternative="less")
        if test.pvalue < 0.05:
            print(f"p-value < 0.05")
        else:
            print("–")
    print()